# R Programming Laboratory - Problem Statement 2
## Comprehensive Lab Solutions: Lab 3 & Lab 4

---

### **Lab 3: Control Flow for Data Cleaning**
* **Topic:** Loops, Functions, and Error Handling in R
* **Dataset:** UCI Heart Disease Dataset (`trestbps`, `chol`, `age`)

### **Lab 4: Advanced Missing Data Handling**
* **Topic:** NA, NULL, NaN, Missing-Value Detection and Imputation
* **Dataset:** UCI Adult / Census Income Dataset (`age`, `workclass`, `education`, `occupation`, `hours_per_week`, `income`)

---

# ==========================================================
# PART 1: LAB 3 - CONTROL FLOW FOR DATA CLEANING
# ==========================================================

### Step 1: Simulate / Load UCI Heart Disease Dataset with Injected Anomalies
We simulate realistic data-entry errors:
1. Negative BP entries (e.g. -120, -135 mmHg)
2. Extreme outlier readings > 300 mmHg (e.g. 310, 340, 360 mmHg)
3. Missing BP observations (`NA`)
4. Zero and missing values in Cholesterol for ratio edge testing

In [ ]:
set.seed(42)
n_patients <- 303

patient_id <- 1:n_patients
age <- round(pmax(29, pmin(77, rnorm(n_patients, mean = 54, sd = 9))))
sex <- sample(c(0, 1), n_patients, replace = TRUE, prob = c(0.32, 0.68))
cp <- sample(0:3, n_patients, replace = TRUE, prob = c(0.47, 0.17, 0.28, 0.08))
trestbps_raw <- round(rnorm(n_patients, mean = 131, sd = 17.5))
chol_raw <- round(pmax(126, pmin(564, rnorm(n_patients, mean = 246, sd = 51))))
thalach <- round(rnorm(n_patients, mean = 149, sd = 23))
target <- sample(c(0, 1), n_patients, replace = TRUE, prob = c(0.46, 0.54))

# Injected anomalies
trestbps_raw[c(15, 78, 142, 220)] <- c(-120, -135, -110, -145) # Negative BP
trestbps_raw[c(34, 112, 189, 275)] <- c(310, 340, 325, 360)    # Extreme BP > 300
trestbps_raw[c(50, 95, 160, 210, 280)] <- NA                    # Missing BP
chol_raw[c(25, 130)] <- 0                                        # Zero chol
chol_raw[c(60, 205)] <- NA                                       # Missing chol

heart_df <- data.frame(patient_id, age, sex, cp, trestbps = trestbps_raw, chol = chol_raw, thalach, target)
head(heart_df, 10)

### Task 1: Create BP-Cleaning Function using if-else
- Detect negative BP and convert to `NA`
- Detect BP > 250 mmHg and cap at `250`
- Retain valid BP without modification

In [ ]:
# Task 1: Single element cleaner
clean_single_bp <- function(bp) {
  if (is.na(bp)) {
    return(NA_real_)
  } else if (bp < 0) {
    return(NA_real_)
  } else if (bp > 250) {
    return(250)
  } else {
    return(as.numeric(bp))
  }
}

# Loop-based vector cleaner
clean_bp_loop <- function(bp_vec) {
  res <- numeric(length(bp_vec))
  for (i in seq_along(bp_vec)) {
    res[i] <- clean_single_bp(bp_vec[i])
  }
  return(res)
}

# Vectorized cleaner
clean_bp_vectorized <- function(bp_vec) {
  res <- bp_vec
  res[res < 0] <- NA_real_
  res[res > 250 & !is.na(res)] <- 250
  return(res)
}

# Test cases
test_vals <- c(-120, 120, 270, NA, 320, 140, -50)
print(sapply(test_vals, clean_single_bp))

### Task 2: Implement Error Handling using tryCatch()
1. `safe_mean_bp`: Safely calculates mean BP when missing/invalid values are present.
2. `safe_calc_ratio`: Safely calculates `chol / trestbps` handling zero denominators, NAs, negative values, and non-numeric inputs.

In [ ]:
# 2.1 Safe Mean BP
safe_mean_bp <- function(bp_vector, na_rm = TRUE) {
  tryCatch(
    expr = {
      if (!is.numeric(bp_vector)) stop("Input must be numeric.")
      if (length(bp_vector) == 0 || all(is.na(bp_vector))) {
        warning("Vector is empty or all NAs.")
        return(NA_real_)
      }
      return(mean(bp_vector, na.rm = na_rm))
    },
    warning = function(w) { cat("[WARNING]:", conditionMessage(w), "\n"); return(NA_real_) },
    error = function(e) { cat("[ERROR]:", conditionMessage(e), "\n"); return(NA_real_) }
  )
}

# 2.2 Safe Ratio Calculation
safe_calc_ratio <- function(chol, trestbps) {
  tryCatch(
    expr = {
      if (!is.numeric(chol) || !is.numeric(trestbps)) stop("Numerator and denominator must be numeric.")
      if (is.na(chol) || is.na(trestbps)) { warning("NA value encountered."); return(NA_real_) }
      if (trestbps <= 0) { warning("Denominator is <= 0."); return(NA_real_) }
      return(round(chol / trestbps, 4))
    },
    warning = function(w) { cat("[HANDLED WARNING]:", conditionMessage(w), "\n"); return(NA_real_) },
    error = function(e) { cat("[HANDLED ERROR]:", conditionMessage(e), "\n"); return(NA_real_) }
  )
}

# Testing edge cases
cat("Safe Mean:", safe_mean_bp(c(120, 130, 140, NA)), "\n")
cat("Ratio (240 / 120):", safe_calc_ratio(240, 120), "\n")
cat("Ratio (240 / 0):", safe_calc_ratio(240, 0), "\n")
cat("Ratio (240 / -120):", safe_calc_ratio(240, -120), "\n")
cat("Ratio ('text' / 120):", safe_calc_ratio("text", 120), "\n")

### Task 3: Compare Loop-based and Vectorized Data Cleaning
We benchmark both methods on $N = 500,000$ values using `system.time()`.

In [ ]:
large_n <- 500000
large_sample <- sample(heart_df$trestbps, size = large_n, replace = TRUE)

t_loop <- system.time({ clean_loop_res <- clean_bp_loop(large_sample) })
t_vec <- system.time({ clean_vec_res <- clean_bp_vectorized(large_sample) })

cat("Loop Elapsed Time:", t_loop["elapsed"], "seconds\n")
cat("Vectorized Elapsed Time:", t_vec["elapsed"], "seconds\n")
cat("Speedup Factor:", round(t_loop["elapsed"] / max(t_vec["elapsed"], 0.0001), 2), "x\n")
cat("Results Match:", all.equal(clean_loop_res, clean_vec_res), "\n")

### Task 4: Validate Cleaned Data & Export

In [ ]:
heart_df$trestbps_cleaned <- clean_bp_vectorized(heart_df$trestbps)
bp_valid <- na.omit(heart_df$trestbps_cleaned)

cat("Missing BP Before:", sum(is.na(heart_df$trestbps)), "\n")
cat("Missing BP After:", sum(is.na(heart_df$trestbps_cleaned)), "\n")
cat("Min BP:", min(bp_valid), "\n")
cat("Max BP:", max(bp_valid), "\n")
cat("Mean BP:", mean(bp_valid), "\n")
cat("Median BP:", median(bp_valid), "\n")
cat("Any Negative remaining?", any(bp_valid < 0), "\n")
cat("Any > 250 remaining?", any(bp_valid > 250), "\n")

# Export
write.csv(heart_df, "cleaned_heart_data.csv", row.names = FALSE)

# ==========================================================
# PART 2: LAB 4 - ADVANCED MISSING DATA HANDLING
# ==========================================================

### Step 1: Simulate UCI Adult Dataset with Multiple Missing Scenarios
Injected patterns:
1. `NA` values in numeric attributes (`age`, `hours_per_week`)
2. `NaN` values in `hours_per_week`
3. Blank strings `""` in categorical features (`workclass`, `occupation`)
4. Impossible values: `age = 999`

In [ ]:
set.seed(123)
n_records <- 500

age_raw <- round(pmax(17, pmin(75, rnorm(n_records, mean = 38.5, sd = 13.5))))
workclass_pool <- c("Private", "Self-emp-not-inc", "Self-emp-inc", "Federal-gov", "Local-gov", "State-gov")
workclass_raw <- sample(workclass_pool, n_records, replace = TRUE, prob = c(0.70, 0.08, 0.04, 0.03, 0.08, 0.07))
education_pool <- c("Bachelors", "Some-college", "11th", "HS-grad", "Prof-school", "Assoc-acdm", "Assoc-voc", "Masters", "Doctorate")
education_raw <- sample(education_pool, n_records, replace = TRUE)
occupation_pool <- c("Tech-support", "Craft-repair", "Other-service", "Sales", "Exec-managerial", "Prof-specialty", "Handlers-cleaners", "Adm-clerical")
occupation_raw <- sample(occupation_pool, n_records, replace = TRUE)
hours_per_week_raw <- round(pmax(1, pmin(99, rnorm(n_records, mean = 40.4, sd = 12.3))))
income_raw <- sample(c("<=50K", ">50K"), n_records, replace = TRUE, prob = c(0.76, 0.24))

# Injected missing & invalid data
age_raw[c(12, 45, 88, 150, 230, 310, 420)] <- NA
hours_per_week_raw[c(20, 65, 110, 195, 275, 360, 480)] <- NA
hours_per_week_raw[c(35, 180, 305)] <- NaN
workclass_raw[c(10, 55, 99, 175, 245, 333, 410, 470)] <- ""
occupation_raw[c(22, 67, 105, 188, 260, 345, 430, 490)] <- ""
age_raw[c(5, 72, 160, 290, 395)] <- 999

adult_df <- data.frame(
  record_id = 1:n_records,
  age = age_raw,
  workclass = workclass_raw,
  education = education_raw,
  occupation = occupation_raw,
  hours_per_week = hours_per_week_raw,
  income = income_raw,
  stringsAsFactors = FALSE
)
head(adult_df, 10)

### Task 1: Identify Different Forms of Missing/Invalid Data & Demonstrate NULL Concept

In [ ]:
cat("is.na() on age:", sum(is.na(adult_df$age)), "\n")
cat("is.nan() on hours_per_week:", sum(is.nan(adult_df$hours_per_week)), "\n")
cat("Blank workclass:", sum(adult_df$workclass == ""), "\n")
cat("Blank occupation:", sum(adult_df$occupation == ""), "\n")
cat("Impossible age (999):", sum(adult_df$age == 999, na.rm = TRUE), "\n")

# NULL Demonstration
d_test <- data.frame(x = 1:3, y = c('a','b','c'))
d_test$y <- NULL # Column 'y' is dropped entirely!
print(names(d_test))

### Task 2 & 3: Custom Median Imputation & Missing-Data Treatment Strategy

In [ ]:
# Custom median imputation function
impute_median <- function(vec) {
  if (!is.numeric(vec)) stop("Vector must be numeric.")
  mask <- is.na(vec) | is.nan(vec)
  val_med <- median(vec[!mask], na.rm = TRUE)
  vec[mask] <- val_med
  return(vec)
}

# Treatment Strategy
adult_df_clean <- adult_df
# 1. Convert 999 to NA
adult_df_clean$age[adult_df_clean$age == 999] <- NA_real_
# 2. Replace blanks with 'Unknown'
adult_df_clean$workclass[adult_df_clean$workclass == ""] <- "Unknown"
adult_df_clean$occupation[adult_df_clean$occupation == ""] <- "Unknown"
# 3. Median Impute numeric features
adult_df_clean$age <- impute_median(adult_df_clean$age)
adult_df_clean$hours_per_week <- impute_median(adult_df_clean$hours_per_week)

cat("Complete cases post-treatment:", sum(complete.cases(adult_df_clean)), "/", nrow(adult_df_clean), "(100%)\n")

### Task 4 & 5: Visualizing Missingness & Validating Dataset

In [ ]:
# Summary validation
cat("=== Post-Cleaning Summary ===\n")
summary(adult_df_clean)

# Export cleaned CSV
write.csv(adult_df_clean, "cleaned_adult_data.csv", row.names = FALSE)
cat("Successfully exported cleaned_adult_data.csv!\n")